# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subata24/ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
%pip install -q duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{}');".format(os.environ["HF_TOKEN"]))

label_df = con.sql("""
WITH feb AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
mar AS (
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_mar
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    mar.client_hash_id,
    mar.content_hash_id,
    feb.clicks_feb,
    mar.clicks_mar,
    CASE WHEN mar.clicks_mar < feb.clicks_feb THEN 1 ELSE 0 END AS declined
FROM mar
JOIN feb ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
""").df()

print(label_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(134238, 5)




## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

##**Method:** Logistic Regression first, then Random Forest.

**Why:** My lane's target (`declined`, 0/1 — did this page's clicks drop month over month) is a yes/no question with an observed label, which the toolkit maps directly to Logistic Regression → Random Forest. Logistic Regression is interpretable — its coefficients can be sanity-checked against what Weeks 3–4 already found (staleness and CTR-underperformance both correlate with decline). Random Forest is tried next as a stronger, still-explainable comparison (via feature importance), without jumping straight to an opaque model the comparison hasn't earned yet.

**Features:** the same signals proven honest in Weeks 3–4, all knowable before March closes:
- `avg_position_feb` — February average ranking position
- `sessions_organic_feb` — February organic sessions
- `engaged_sessions_feb` — February engaged sessions
- `word_count` — static content length
- `search_volume` — static keyword-market attribute
- `content_age_days` — staleness signal (Week-4 Signal 1, CONFIRMED)
- `ctr_gap` — CTR-vs-expected-position gap (Week-4 Signal 2, MIXED but directionally useful)

None of these are derived from March's own outcome columns — same exclusion rule as the Week-3 data contract.

In [13]:
con.register("label_df_tbl", label_df)

features_df = con.sql("""
WITH feb_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS sessions_organic_feb,
        SUM(ga4_engaged_sessions) AS engaged_sessions_feb,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    fm.client_hash_id,
    fm.content_hash_id,
    fm.avg_position_feb,
    fm.sessions_organic_feb,
    fm.engaged_sessions_feb,
    dc.word_count,
    dc.search_volume,
    DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days,
    CASE WHEN fm.feb_impressions > 0 THEN fm.feb_clicks * 1.0 / fm.feb_impressions ELSE NULL END AS ctr,
    CASE
        WHEN fm.avg_position_feb <= 3 THEN 0.25
        WHEN fm.avg_position_feb <= 10 THEN 0.05
        ELSE 0.01
    END AS expected_ctr,
    fm.feb_impressions
FROM feb_metrics fm
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' dc
    ON fm.client_hash_id = dc.client_hash_id AND fm.content_hash_id = dc.content_hash_id
WHERE dc.content_created_date IS NOT NULL
""").df()

features_df["ctr_gap"] = features_df["ctr"] - features_df["expected_ctr"]

print(features_df.shape)
features_df.isna().sum()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(153559, 12)


,0
client_hash_id,0
content_hash_id,0
avg_position_feb,0
sessions_organic_feb,81138
engaged_sessions_feb,81138
word_count,50994
search_volume,17476
content_age_days,0
ctr,0
expected_ctr,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

##**Split: grouped by client_hash_id (GroupShuffleSplit, 70/30 target, seed=42).**

A time-aware split isn't available here — the label is a single Feb→March snapshot, not a rolling series, so there's only one before/after boundary to split on, and it's already fixed by the label definition itself.

Grouping by client matters more: a random row split would let the same client's other pages appear in both train and test, letting the model partly learn "this specific client behaves this way" rather than a signal that generalizes to a client it has never seen. Since FlyRank's real deployment scenario is scoring pages for clients the model may not have trained on, grouped-by-client is the honest split for this question.

**Result:** 26 clients in train (40,759 rows), 12 clients in test (34,684 rows), zero client overlap between train and test (confirmed programmatically). The row split landed at roughly 54%/46% rather than the requested 70/30 — an expected side effect of splitting by client count rather than row count, since client page-counts vary widely. This is reported honestly rather than adjusted to hit an exact ratio, since forcing an exact 70/30 row split would require either breaking client grouping or manually re-weighting, both of which would reduce the honesty of the split.

In [14]:
from sklearn.model_selection import GroupShuffleSplit

merged = label_df.merge(features_df, on=["client_hash_id", "content_hash_id"])
merged = merged.dropna(subset=["avg_position_feb", "word_count", "search_volume"])

feature_cols = ["avg_position_feb", "sessions_organic_feb", "engaged_sessions_feb",
                "word_count", "search_volume", "content_age_days", "ctr_gap"]

X = merged[feature_cols].fillna(0)
y = merged["declined"]
groups = merged["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train clients:", merged.iloc[train_idx]['client_hash_id'].nunique())
print("Test clients:", merged.iloc[test_idx]['client_hash_id'].nunique())
print("Overlap check (should be 0):", len(set(merged.iloc[train_idx]['client_hash_id']) & set(merged.iloc[test_idx]['client_hash_id'])))

Train: (40759, 7) Test: (34684, 7)
Train clients: 26
Test clients: 12
Overlap check (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

All three scored on the same grouped train/test split (26 train clients, 12 test clients, zero overlap), same feature set, same test rows.

**AUC:**

| Method | AUC |
|---|---|
| Baseline (Week-4 rule) | 0.633 |
| Logistic Regression | 0.629 |
| Random Forest | 0.843 |

**Precision@K** (base rate in test = 0.217):

| K | Baseline | Logistic Regression | Random Forest |
|---|---|---|---|
| 20 | 0.30 | 0.50 | 0.90 |
| 50 | 0.18 | 0.48 | 0.84 |
| 100 | 0.29 | 0.45 | 0.76 |

Random Forest outperforms both the baseline and Logistic Regression on every metric at every K. However, this win needs a caveat before it's trusted at face value — see Section 4: the size of this gap points to the model partly exploiting a mechanical property of the label rather than pure content-health signal.

Step 1: score the baseline rule on the test set.

In [15]:
import numpy as np

merged_test = merged.iloc[test_idx].copy()

# Reconstruct the Week-4 baseline flags on the test set
merged_test["stale_flag"] = (merged_test["content_age_days"] >= 90).astype(int)
merged_test["low_ctr_flag"] = np.where(
    merged_test["ctr_gap"].isna(), 0,
    (merged_test["ctr_gap"] < -0.02).astype(int)
)
merged_test["baseline_score"] = merged_test["stale_flag"] + merged_test["low_ctr_flag"]

from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(y_test, merged_test["baseline_score"])
print("Baseline AUC:", baseline_auc)

Baseline AUC: 0.6326658659497312


Step 2: train Logistic Regression and Random Forest on the same train/test split.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_auc = roc_auc_score(y_test, logreg.predict_proba(X_test)[:, 1])

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

print("Logistic Regression AUC:", logreg_auc)
print("Random Forest AUC:", rf_auc)
print("Baseline AUC:", baseline_auc)

Logistic Regression AUC: 0.6293224578077194
Random Forest AUC: 0.8430858756712205
Baseline AUC: 0.6326658659497312


In [17]:
import pandas as pd

importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(importances)

                feature  importance
6               ctr_gap    0.400454
1  sessions_organic_feb    0.185200
5      content_age_days    0.160032
0      avg_position_feb    0.116826
3            word_count    0.082014
2  engaged_sessions_feb    0.039243
4         search_volume    0.016231


In [18]:
print(merged.groupby(merged["clicks_feb"] == 0)["declined"].mean())
print(merged["clicks_feb"].describe())

clicks_feb
False    0.404175
True     0.000000
Name: declined, dtype: float64
count    75443.000000
mean         6.409157
std         29.555581
min          0.000000
25%          0.000000
50%          1.000000
75%          4.000000
max       3310.000000
Name: clicks_feb, dtype: float64


In [19]:
def precision_at_k(y_true, y_scores, k):
    order = np.argsort(-y_scores)
    top_k = order[:k]
    return y_true.iloc[top_k].mean()

logreg_scores = logreg.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]
baseline_scores = merged_test["baseline_score"].values

results = []
for k in [20, 50, 100]:
    results.append({
        "k": k,
        "baseline_precision": precision_at_k(y_test, baseline_scores, k),
        "logreg_precision": precision_at_k(y_test, logreg_scores, k),
        "rf_precision": precision_at_k(y_test, rf_scores, k),
    })

precision_table = pd.DataFrame(results)
print(precision_table)

print("\nBase rate (overall decline rate in test):", y_test.mean())
print("\nAUC — Baseline:", baseline_auc, "| LogReg:", logreg_auc, "| RF:", rf_auc)

     k  baseline_precision  logreg_precision  rf_precision
0   20                0.30              0.50          0.90
1   50                0.18              0.48          0.84
2  100                0.29              0.45          0.76

Base rate (overall decline rate in test): 0.2171318187060316

AUC — Baseline: 0.6326658659497312 | LogReg: 0.6293224578077194 | RF: 0.8430858756712205


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### What the model leans on

The Random Forest relied most heavily on **ctr_gap**, which accounted for about 40% of the total feature importance. February organic sessions and content age were the next strongest predictors, while search volume contributed very little.

| Feature | Importance |
|---|---:|
| ctr_gap | 0.400 |
| sessions_organic_feb | 0.185 |
| content_age_days | 0.160 |
| avg_position_feb | 0.117 |
| word_count | 0.082 |
| engaged_sessions_feb | 0.039 |
| search_volume | 0.016 |

This broadly agrees with the Week-4 findings, where CTR underperformance and content age were identified as useful indicators of future decline.

During error analysis, I also noticed an important property of the target label. Pages with `clicks_feb = 0` can never receive a decline label because click counts cannot decrease below zero. This is not data leakage because every feature is still measured before the March outcome, but it creates a floor effect in the label. As a result, part of the Random Forest's improvement may come from learning this property of the target rather than only learning genuine content-decay patterns.

### Error pattern

Using the default probability threshold of 0.5, the Random Forest produced:

- **False positives:** 209
- **False negatives:** 7,128

The model misses many more declining pages than it incorrectly flags. This behaviour suggests that the classifier is conservative at the default threshold and often predicts that a page will not decline unless the evidence is relatively strong.

### Example errors

**False positive**

- `content_a6f0f0e0b464c9d2`
  - February clicks: 9
  - March clicks: 11
  - Predicted probability: 0.508
  - The model predicted a decline with only slightly more than 50% confidence, but the page gained clicks instead. This appears to be a borderline decision where the model slightly overestimated the risk.

**False negative**

- `content_71c9a463c85da45c`
  - February clicks: 2
  - March clicks: 0
  - Predicted probability: 0.165
  - Despite the page declining completely, the model assigned a very low probability of decline, indicating that other features outweighed the negative CTR signal.

**False negative**

- `content_d03a50212e6f6823`
  - February clicks: 23
  - March clicks: 16
  - Predicted probability: 0.495
  - The prediction was very close to the decision threshold but remained below 0.5, showing that some mistakes are borderline cases rather than confident incorrect predictions.

### Takeaway

The Random Forest substantially outperformed both the Week-4 baseline rule and Logistic Regression on the grouped client-level test split. However, the results should be interpreted with care because the target label contains a natural floor effect for pages with zero February clicks. A useful follow-up experiment would be to repeat the analysis after restricting the dataset to pages with at least one February click and check whether the same feature rankings and performance improvements remain.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
test_results = merged_test.copy()
test_results["rf_pred_proba"] = rf_scores
test_results["rf_pred"] = (rf_scores >= 0.5).astype(int)
test_results["actual"] = y_test.values

# False positives: model said declined, but it didn't
false_positives = test_results[(test_results["rf_pred"] == 1) & (test_results["actual"] == 0)]
# False negatives: model said fine, but it declined
false_negatives = test_results[(test_results["rf_pred"] == 0) & (test_results["actual"] == 1)]

print("False positives:", len(false_positives), "| False negatives:", len(false_negatives))
print("\n--- 3 false positive examples ---")
print(false_positives[["content_hash_id", "clicks_feb", "clicks_mar", "ctr_gap", "content_age_days", "rf_pred_proba"]].head(3))
print("\n--- 3 false negative examples ---")
print(false_negatives[["content_hash_id", "clicks_feb", "clicks_mar", "ctr_gap", "content_age_days", "rf_pred_proba"]].head(3))

False positives: 209 | False negatives: 7128

--- 3 false positive examples ---
               content_hash_id  clicks_feb  clicks_mar   ctr_gap  \
3145  content_a6f0f0e0b464c9d2         9.0        11.0 -0.002623   
3625  content_eb578906bb84a43b         4.0         4.0 -0.003139   
3849  content_66c92984fffb9402         6.0         7.0 -0.004690   

      content_age_days  rf_pred_proba  
3145               363       0.508416  
3625               233       0.510204  
3849               210       0.533017  

--- 3 false negative examples ---
               content_hash_id  clicks_feb  clicks_mar   ctr_gap  \
28    content_71c9a463c85da45c         2.0         0.0 -0.048695   
69    content_d18ed766938f885c         1.0         0.0 -0.232143   
3074  content_d03a50212e6f6823        23.0        16.0 -0.039297   

      content_age_days  rf_pred_proba  
28                  17       0.165323  
69                  11       0.118251  
3074               360       0.494768  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.